# Epi4Ab Comprehensive Training & Debugging Notebook

This notebook provides comprehensive training, debugging, and model generation capabilities for the Epi4Ab epitope prediction system.

## Features Available:
- **Standard 3-fold Cross-Validation Training** (train_all = no)
- **Full Model Generation** (train_all = yes) for inference
- **Hyperparameter Optimization** with Optuna
- **Model Testing & Evaluation** 
- **Path Diagnostics** for troubleshooting
- **CDR Sequence Validation**

## Important Notes:
- Use `train_all = no` for research/validation (3-fold CV, no model.pt saved)
- Use `train_all = yes` for production model generation (saves model.pt for inference)
- Check paths if you encounter FileNotFound errors
- Verify CDR sequences are properly generated

In [ ]:
# System Diagnostics & Path Verification
import os
import sys
from pathlib import Path

print("🔍 System Diagnostics")
print("=" * 50)

# Check current working directory
print(f"📁 Current Directory: {os.getcwd()}")

# Check if key files exist
key_files = ['.env', 'parameters.txt', 'data/sabdab_summary_master.tsv', 'lists/train_pdb.txt', 'lists/test_pdb.txt']
for file in key_files:
    status = "✅" if os.path.exists(file) else "❌"
    print(f"{status} {file}")

# Check data directory
data_dir = "data/nodes_edges"
if os.path.exists(data_dir):
    pdb_count = len([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    print(f"📊 Found {pdb_count} PDB structures in {data_dir}")
else:
    print(f"❌ Data directory {data_dir} not found")

# Check CDR sequences
sample_cdr = "data/nodes_edges/1zmy/sequence/cdr_sequence.json"
if os.path.exists(sample_cdr):
    print("✅ CDR sequences appear to be generated")
else:
    print("❌ CDR sequences may need generation")

print("\n🎯 Ready for training!")
print("=" * 50)

In [ ]:
# Training Configuration Options
print("⚙️ Training Configuration")
print("=" * 50)

# Training mode selection
TRAINING_MODE = "cross_validation"  # Options: "cross_validation", "full_model", "debug"

if TRAINING_MODE == "cross_validation":
    print("🔄 Mode: 3-Fold Cross-Validation")
    print("   - Best for research and validation")
    print("   - Provides robust performance metrics")
    print("   - No model.pt file saved")
    train_all_setting = "no"
    epochs = 50
    
elif TRAINING_MODE == "full_model":
    print("🏗️ Mode: Full Model Generation")
    print("   - Trains on all available data")
    print("   - Saves model.pt for inference")
    print("   - Use for production deployment")
    train_all_setting = "yes"
    epochs = 50
    
elif TRAINING_MODE == "debug":
    print("🐛 Mode: Debug (Quick Test)")
    print("   - Fast training for testing")
    print("   - Reduced epochs for quick validation")
    train_all_setting = "no"
    epochs = 2

print(f"📋 Settings: train_all={train_all_setting}, epochs={epochs}")
print("=" * 50)

# Epi4Ab Training Notebook
Interactive training interface for the Epi4Ab epitope prediction model

In [ ]:
print('''------ AMPM Project ------
*** Training Model ***
''')

# Suppress warnings for clean output
import warnings
warnings.filterwarnings("ignore")
import os
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
from datetime import datetime

print('🚀 Importing script modules...')
import_script_start = datetime.now()

try:
    from interface_prediction.run_setup.logging import ModelLogging
    from interface_prediction.run_setup.set_up import create_output_folder
    from interface_prediction.run_setup.arguments import initiate_argument
    from interface_prediction.data_function.data_function import process_data
    from interface_prediction.model.training_function import process_training
    from interface_prediction.evaluation_and_plot.result_evaluation import table_mean
    from interface_prediction.evaluation_and_plot.loss import result_loss
    print('✅ All modules imported successfully')
except ImportError as e:
    print(f'❌ Import error: {e}')
    print('💡 Make sure you are in the correct directory and environment is activated')
    raise

import_script_time = datetime.now() - import_script_start
print(f'⏱️ Import time: {import_script_time}')

In [ ]:
prepare_start_time = datetime.now()
args = initiate_argument()
# Prepare folder

if os.path.isdir(args.directory_output) == False:
    os.mkdir(args.directory_output)

model_logging = ModelLogging(args)
setup_logging_time = datetime.now() - prepare_start_time
print(f'Set up logging: {setup_logging_time}')

folder_name = str(model_logging.run_date) + '_' + model_logging.model_name
output_folder = create_output_folder(folder_name, model_logging.directory_output)
model_logging.directory_output_folder = output_folder
test_record_folder = os.path.join(output_folder,'test_record')
os.mkdir(test_record_folder)
model_logging.directory_test_record = test_record_folder
print(f'Results are located in {output_folder}.')
model_logging.prepare_time = datetime.now() - prepare_start_time

In [ ]:
data_start_time = datetime.now()
if model_logging.train_all == 'yes':
    train_data, train_list = process_data(model_logging)
else:
    train_data, train_list, relaxed_train_data, relaxed_train_list, af_train_data, adj_af_train_list, test_data, test_list = process_data(model_logging)
model_logging.data_time = datetime.now() - data_start_time

In [ ]:
print('Start training')

train_start_time = datetime.now()
if model_logging.train_all == 'yes':
    train_loss_record, evaluation_record = process_training(train_data, train_list, model_logging)
else:
    train_loss_record, evaluation_record = process_training(train_data, train_list, model_logging, relaxed_train_data, relaxed_train_list, 
                                                            af_train_data, adj_af_train_list, test_data, test_list)
model_logging.train_time = datetime.now() - train_start_time

In [ ]:
result_start_time = datetime.now()
# Loss
if model_logging.train_all in ['yes','with_validation']:
    info_loss_columns = ['type', 'epoch', 'loss']
else:
    info_loss_columns = ['type', 'epoch', 'fold', 'loss']
loss_df = pd.DataFrame(train_loss_record, columns = info_loss_columns)
# Evaluation
if model_logging.train_all in ['yes','with_validation']:
    info_record_columns = ['pdbId', 'Model']
else:
    info_record_columns = ['pdbId','Fold','Model']
record_columns = info_record_columns + model_logging.evaluation_columns + ['Type'] + ['Label']
model_logging.info_record_columns = record_columns
evaluation_df = pd.DataFrame(evaluation_record, columns = record_columns)

In [ ]:
print('Evaluating result.')
plotting_loss_start_time = datetime.now()
result_loss(loss_df, model_logging)
plotting_loss_time = datetime.now() - plotting_loss_start_time
print(f'Plotting loss: {plotting_loss_time}')

generate_table_start_time = datetime.now()
table_mean(evaluation_df, model_logging, resultRegion=model_logging.use_region)
generate_table_time = datetime.now() - generate_table_start_time
print(f'Summarize average result: {generate_table_time}')
model_logging.result_time = datetime.now() - result_start_time

In [ ]:
model_logging.log_result()
print('Generate log: Done')

with open(os.path.join(output_folder,'parameters.txt'), 'w') as f:
    f.write(f'# Code version: {model_logging.code_version}\n')
    f.write(args.saved_params.replace(';','\n').replace('+',' '))
print('Saving parameters: Done')